# test_model: Gurobi sizing model (Gavin's sandbox)
Separate from the team's workflows. Combines **Patrick's** heat pump physics (`model.ipynb`: heating load from UA and internal gains, COP and capacity curves vs outdoor temperature, heat pump first and gas backup for the shortfall) with **Sharvi's** installed cost (`equation.py`: `68.09 * kBtu/h + 5100`) inside a Gurobi optimization that **chooses the heat pump size** (or keeps gas) to minimize

`up-front cost - rebate + years x (energy bill) + carbon price x years x CO2`

Sweeping the carbon price traces the cost-vs-CO2 trade-off, like the dashboard's dial.

**Design notes**
- The Gurobi pip license caps a model at 2,000 variables/constraints, so the year is compressed into outdoor-temperature bins (5 F wide) with their hours. Each bin is a steady-state heat balance. That is ~50 variables.
- Patrick's hourly RC simulation (copied unchanged below) is then run at the chosen size on the real hourly weather as a cross-check of the compressed model.
- Minneapolis only (the data in `data/processed/`). Home is Patrick's default (3,310 sq ft, R 5.065).
- Run from the repo root. Needs `pip install gurobipy pandas numpy scipy matplotlib`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB
from scipy.interpolate import interp1d

## Inputs (sources noted; change here)

In [ ]:
# --- Home (Patrick's defaults in run_thermal_hp_sim) ---
AREA_SQFT   = 3310.0
R_VAL       = 5.065            # unified envelope R
SETPOINT_F  = 68.0
Q_INT_BTUH  = 2000.0           # internal gains, Btu/h (middle of Patrick's 1200-2800 mock range)
UA          = AREA_SQFT / R_VAL  # Btu/h/F

# --- Heat pump (Patrick: 47/17/5/-22 F curves; capacity scaled from nominal as in his parameter search) ---
COP_PTS = {47.0: 3.60, 17.0: 2.40, 5.0: 1.80, -22.0: 1.00}
CAP_FRAC = {47.0: 1.00, 17.0: 0.80, 5.0: 0.65, -22.0: 0.35}   # fraction of nominal kBtu/h
MIN_KBTUH, MAX_KBTUH = 18, 60  # valid range of Sharvi's cost equation

# --- Cost (Sharvi, equation.py: NREL regression, 2023 $, before rebates) ---
def capital_cost(kbtuh): return 68.09 * kbtuh + 5100
REBATE_USD = 2000              # Xcel ccASHP rebate (model_inputs_minneapolis.json); CenterPoint $1,100 not applied

# --- Prices / emissions (data/processed/model_inputs_minneapolis.json) ---
ELEC_USD_KWH = 0.1644
GAS_USD_KWH_FUEL = 0.035
FURNACE_AFUE = 0.95
GRID_KG_KWH = 0.4173           # eGRID MROW annual; repo MISO hourly file averages ~0.435
GAS_KG_KWH_FUEL = 0.181
YEARS = 15                     # no discounting, like model v1
BTU_PER_KWH = 3412.14

## Weather and load: real hourly TMYx, binned by outdoor temperature

In [ ]:
w = pd.read_csv("data/processed/msp_weather_TMYx.2011-2025.csv")
w["t_f"] = w["drybulb_C"] * 9/5 + 32
w["load_btuh"] = np.maximum(0.0, UA * (SETPOINT_F - w["t_f"]) - Q_INT_BTUH)

BIN_F = 5
w["bin_f"] = (np.floor(w["t_f"] / BIN_F) * BIN_F + BIN_F / 2)      # bin centre
bins = (w.groupby("bin_f").agg(hours=("t_f", "size"), load_btuh=("load_btuh", "mean")).reset_index())
bins = bins[bins["load_btuh"] > 0].reset_index(drop=True)          # only bins with a heating need

cop_f = interp1d(sorted(COP_PTS), [COP_PTS[k] for k in sorted(COP_PTS)], bounds_error=False,
                 fill_value=(COP_PTS[min(COP_PTS)], COP_PTS[max(COP_PTS)]))
cap_f = interp1d(sorted(CAP_FRAC), [CAP_FRAC[k] for k in sorted(CAP_FRAC)], bounds_error=False,
                 fill_value=(CAP_FRAC[min(CAP_FRAC)], CAP_FRAC[max(CAP_FRAC)]))
bins["cop"] = cop_f(bins["bin_f"]).astype(float)
bins["cap_frac"] = cap_f(bins["bin_f"]).astype(float)
bins["load_kwh_yr"] = bins["load_btuh"] * bins["hours"] / BTU_PER_KWH
print(f"{len(bins)} bins, annual heat need {bins['load_kwh_yr'].sum():,.0f} kWh, peak hourly load {w['load_btuh'].max()/1000:.1f} kBtu/h")
bins.round(2)

## The Gurobi model
Per bin *b* (hours `H_b`, average load `L_b` in Btu/h, COP `c_b`, capacity fraction `f_b`):

- `h_b + a_b = L_b` (heat pump heat plus gas backup heat meets the load)
- `h_b <= 1000 * f_b * s` (heat pump limited by its capacity at that temperature)
- `18 y <= s <= 60 y`, `y` binary (install or keep gas)
- `dispatch="hp_first"` adds a binary per bin so that `h_b = min(L_b, capacity)` (Patrick's rule); `"optimal"` leaves `h_b` free
- annual kWh electric `= sum H_b h_b / (c_b 3412)`; annual gas fuel `= sum H_b a_b / (AFUE 3412)`

Objective: `y*(capex(s)-rebate)` is linear because `capex = 68.09 s + 5100` and the rebate only applies if installed.

In [ ]:
def solve(lam_usd_per_t, dispatch="optimal", bins=bins, verbose=False):
    # dispatch="optimal": Gurobi also chooses, hour-bin by hour-bin, how much heat comes from the heat pump vs gas (a smart controller).
    # dispatch="hp_first": Patrick's rule, the heat pump carries as much as its capacity allows and gas covers only the shortfall.
    m = gp.Model("sizing"); m.Params.OutputFlag = 1 if verbose else 0
    s = m.addVar(lb=0, ub=MAX_KBTUH, name="size_kbtuh")
    y = m.addVar(vtype=GRB.BINARY, name="install")
    h = m.addVars(len(bins), lb=0, name="hp_heat")
    a = m.addVars(len(bins), lb=0, name="aux_heat")
    m.addConstr(s >= MIN_KBTUH * y); m.addConstr(s <= MAX_KBTUH * y)
    for i, r in bins.iterrows():
        m.addConstr(h[i] + a[i] == r.load_btuh)
        m.addConstr(h[i] <= 1000 * r.cap_frac * s)
        if dispatch == "hp_first":      # h = min(load, capacity), linearized with a binary per bin (big-M)
            z = m.addVar(vtype=GRB.BINARY); M = 100000
            m.addConstr(h[i] >= 1000 * r.cap_frac * s - M * (1 - z)); m.addConstr(1000 * r.cap_frac * s <= r.load_btuh + M * (1 - z))
            m.addConstr(h[i] >= r.load_btuh - M * z);                 m.addConstr(1000 * r.cap_frac * s >= r.load_btuh - M * z)
    elec_kwh = gp.quicksum(r.hours * h[i] / (r.cop * BTU_PER_KWH) for i, r in bins.iterrows())
    gas_kwh  = gp.quicksum(r.hours * a[i] / (FURNACE_AFUE * BTU_PER_KWH) for i, r in bins.iterrows())
    opex = ELEC_USD_KWH * elec_kwh + GAS_USD_KWH_FUEL * gas_kwh
    co2_kg = GRID_KG_KWH * elec_kwh + GAS_KG_KWH_FUEL * gas_kwh
    capex = 68.09 * s + (5100 - REBATE_USD) * y
    m.setObjective(capex + YEARS * opex + lam_usd_per_t / 1000 * YEARS * co2_kg, GRB.MINIMIZE)
    m.optimize()
    size = s.X if y.X > 0.5 else 0.0
    e, g = elec_kwh.getValue(), gas_kwh.getValue()
    load = (bins.load_btuh * bins.hours).sum() / BTU_PER_KWH
    hp_heat = sum(r.hours * h[i].X for i, r in bins.iterrows()) / BTU_PER_KWH
    return dict(dispatch=dispatch, lambda_usd_per_t=lam_usd_per_t, size_kbtuh=size, size_tons=size / 12,
                upfront_usd=(capital_cost(size) - REBATE_USD) if size else 0.0,
                annual_cost_usd=ELEC_USD_KWH * e + GAS_USD_KWH_FUEL * g, annual_co2_t=(GRID_KG_KWH * e + GAS_KG_KWH_FUEL * g) / 1000,
                annual_elec_kwh=e, annual_gas_kwh_fuel=g, hp_share=hp_heat / load)

gas_only_cost = GAS_USD_KWH_FUEL * bins.eval("hours*load_btuh").sum() / (FURNACE_AFUE * BTU_PER_KWH)
gas_only_co2 = GAS_KG_KWH_FUEL * bins.eval("hours*load_btuh").sum() / (FURNACE_AFUE * BTU_PER_KWH) / 1000
print(f"Keep gas furnace: ${gas_only_cost:,.0f}/yr, {gas_only_co2:.1f} t CO2/yr")

## Sweep the carbon price (same $/tonne grid as model v1)

In [ ]:
LAMS = [0, 10, 25, 50, 75, 100, 150, 200, 300, 400, 500, 750, 1000]
res = pd.DataFrame([solve(l, d) for d in ("optimal", "hp_first") for l in LAMS])
res["saves_usd_yr"] = gas_only_cost - res["annual_cost_usd"]
res["payback_yr"] = np.where(res.saves_usd_yr > 0, res.upfront_usd / res.saves_usd_yr, np.nan)
res["co2_cut_t_yr"] = gas_only_co2 - res["annual_co2_t"]
res["usd_per_t_avoided"] = np.where(res.co2_cut_t_yr > 0.01,
    (res.upfront_usd + YEARS * (res.annual_cost_usd - gas_only_cost)) / (YEARS * res.co2_cut_t_yr), np.nan)
res.round(2)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for d, g in res.groupby("dispatch"): ax[0].step(g.lambda_usd_per_t, g.size_tons, where="post", marker="o", label=d)
ax[0].legend(); ax[0].set_xlabel("carbon price ($/t CO2)"); ax[0].set_ylabel("chosen size (tons)"); ax[0].set_title("Recommended size vs carbon price")
life_cost = res.upfront_usd + YEARS * res.annual_cost_usd
for d, g in res.assign(life_cost=life_cost).groupby("dispatch"): ax[1].plot(YEARS * g.annual_co2_t, g.life_cost, "o-", label=d)
ax[1].plot([YEARS*gas_only_co2], [YEARS*gas_only_cost], "k*", ms=12, label="keep gas")
ax[1].set_xlabel("15-year CO2 (t)"); ax[1].set_ylabel("15-year cost ($)"); ax[1].set_title("Cost vs carbon"); ax[1].legend()
plt.tight_layout(); plt.show()

## Cross-check: Patrick's hourly RC simulation (unchanged) at the chosen size, on the real hourly weather

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d


def run_thermal_hp_sim(
    df_hourly: pd.DataFrame,
    area_sqft: float = 3310.0,
    r_val: float = 5.065,
    setpoint_f: float = 68.0,
    deadband_f: float = 1.0,
    # Standard multi-point COP ratings dict: {temp_f: COP}
    cop_ratings: dict[float, float] | None = None,
    # Standard multi-point capacity ratings dict: {temp_f: max_btu_per_hr}
    capacity_ratings: dict[float, float] | None = None,
    hp_capacity_tons: float = 3.0,  # Used as fallback baseline if capacity_ratings is None
    gas_furnace_eff: float = 0.80,
    elec_rate_per_kwh: float = 0.15,
    gas_rate_per_therm: float = 1.20,
    grid_co2_kg_per_kwh: float = 0.45,
    gas_co2_kg_per_therm: float = 5.30,
) -> tuple[pd.DataFrame, dict]:
    """Runs an hourly 1-Node RC thermal simulation for a heat pump home with gas backup,

    using 1D linear interpolation for cold-climate COP and capacity curves.
    """
    # 1. Provide default NEEP/AHRI benchmark rating curves if none supplied
    if cop_ratings is None:
        # Standard cold-climate inverter profile (47°F, 17°F, 5°F, -22°F)
        cop_ratings = {47.0: 3.60, 17.0: 2.40, 5.0: 1.80, -22.0: 1.00}

    if capacity_ratings is None:
        # Base 3-ton unit (36,000 BTU/hr nominal at 47°F) with cold weather capacity drop
        capacity_ratings = {
            47.0: hp_capacity_tons * 12000.0,
            17.0: hp_capacity_tons * 12000.0 * 0.78,
            5.0: hp_capacity_tons * 12000.0 * 0.60,
            -22.0: hp_capacity_tons * 12000.0 * 0.30,
        }

    # Sort dictionary keys to ensure ascending arrays for interpolation
    temps_cop = np.array(sorted(cop_ratings.keys()))
    vals_cop = np.array([cop_ratings[t] for t in temps_cop])

    temps_cap = np.array(sorted(capacity_ratings.keys()))
    vals_cap = np.array([capacity_ratings[t] for t in temps_cap])

    # 2. Build SciPy linear interpolation functions
    get_cop_interp = interp1d(
        temps_cop,
        vals_cop,
        kind='linear',
        bounds_error=False,
        fill_value=(vals_cop[0], vals_cop[-1]),  # Clip at boundaries
    )

    get_cap_interp = interp1d(
        temps_cap,
        vals_cap,
        kind='linear',
        bounds_error=False,
        fill_value=(vals_cap[0], vals_cap[-1]),  # Clip at boundaries
    )

    # 3. Setup Building & Simulation State
    n_hours = len(df_hourly)
    ua_btu = area_sqft / r_val
    c_btu = area_sqft * 15.0

    t_in = np.zeros(n_hours)
    q_hp_delivered = np.zeros(n_hours)
    q_aux_delivered = np.zeros(n_hours)
    kwh_hp = np.zeros(n_hours)
    therms_aux = np.zeros(n_hours)
    hvac_action = ['OFF'] * n_hours

    t_in[0] = setpoint_f
    dt_hr = 1.0

    # 4. Hourly Simulation Loop
    for t in range(n_hours - 1):
        t_amb = df_hourly['t_amb_f'].iloc[t]
        q_int = df_hourly['q_int_btu'].iloc[t]

        heat_loss = ua_btu * (t_in[t] - t_amb)
        heat_needed = max(0.0, heat_loss - q_int)

        if t_in[t] < (setpoint_f - deadband_f) or heat_needed > 0:
            # Look up interpolated COP and max capacity for current outdoor temp
            cop_current = float(get_cop_interp(t_amb))
            avail_hp_cap = float(get_cap_interp(t_amb))

            # Heat delivered split
            hp_heat = min(heat_needed, avail_hp_cap)
            aux_heat = max(0.0, heat_needed - hp_heat)

            # Energy inputs
            hp_elec_kwh = (hp_heat / max(1.0, cop_current)) / 3412.14
            gas_therms = (aux_heat / gas_furnace_eff) / 100000.0

            if hp_heat > 0 and aux_heat > 0:
                hvac_action[t] = 'HP_AND_GAS_AUX'
            elif hp_heat > 0:
                hvac_action[t] = 'HP_ONLY'
            elif aux_heat > 0:
                hvac_action[t] = 'GAS_AUX_ONLY'
        else:
            hp_heat = 0.0
            aux_heat = 0.0
            hp_elec_kwh = 0.0
            gas_therms = 0.0
            hvac_action[t] = 'OFF'

        q_hp_delivered[t] = hp_heat
        q_aux_delivered[t] = aux_heat
        kwh_hp[t] = hp_elec_kwh
        therms_aux[t] = gas_therms

        # State update
        net_heat_gain = hp_heat + aux_heat + q_int - heat_loss
        t_in[t + 1] = t_in[t] + (net_heat_gain / c_btu) * dt_hr

    hvac_action[-1] = hvac_action[-2]

    # 5. Format Output Dataframe
    df_res = df_hourly.copy()
    df_res['t_in_f'] = t_in
    df_res['hvac_action'] = hvac_action
    df_res['q_hp_btu'] = q_hp_delivered
    df_res['q_aux_btu'] = q_aux_delivered
    df_res['kwh_hp'] = kwh_hp
    df_res['therms_aux'] = therms_aux

    df_res['elec_cost_usd'] = df_res['kwh_hp'] * elec_rate_per_kwh
    df_res['gas_cost_usd'] = df_res['therms_aux'] * gas_rate_per_therm
    df_res['total_cost_usd'] = (
        df_res['elec_cost_usd'] + df_res['gas_cost_usd']
    )

    df_res['co2_elec_kg'] = df_res['kwh_hp'] * grid_co2_kg_per_kwh
    df_res['co2_gas_kg'] = df_res['therms_aux'] * gas_co2_kg_per_therm
    df_res['total_co2_kg'] = df_res['co2_elec_kg'] + df_res['co2_gas_kg']

    summary = {
        'total_elec_kwh': df_res['kwh_hp'].sum(),
        'total_gas_therms': df_res['therms_aux'].sum(),
        'total_cost_usd': df_res['total_cost_usd'].sum(),
        'total_co2_kg': df_res['total_co2_kg'].sum(),
        'hp_coverage_pct': (
            df_res['q_hp_btu'].sum()
            / max(1.0, (df_res['q_hp_btu'].sum() + df_res['q_aux_btu'].sum()))
        )
        * 100,
        'min_indoor_temp_f': df_res['t_in_f'].min(),
    }

    return df_res, summary

In [ ]:
SIM_SOURCE = 'import numpy as np\nimport pandas as pd\nfrom scipy.interpolate import interp1d\n\n\ndef run_thermal_hp_sim(\n    df_hourly: pd.DataFrame,\n    area_sqft: float = 3310.0,\n    r_val: float = 5.065,\n    setpoint_f: float = 68.0,\n    deadband_f: float = 1.0,\n    # Standard multi-point COP ratings dict: {temp_f: COP}\n    cop_ratings: dict[float, float] | None = None,\n    # Standard multi-point capacity ratings dict: {temp_f: max_btu_per_hr}\n    capacity_ratings: dict[float, float] | None = None,\n    hp_capacity_tons: float = 3.0,  # Used as fallback baseline if capacity_ratings is None\n    gas_furnace_eff: float = 0.80,\n    elec_rate_per_kwh: float = 0.15,\n    gas_rate_per_therm: float = 1.20,\n    grid_co2_kg_per_kwh: float = 0.45,\n    gas_co2_kg_per_therm: float = 5.30,\n) -> tuple[pd.DataFrame, dict]:\n    """Runs an hourly 1-Node RC thermal simulation for a heat pump home with gas backup,\n\n    using 1D linear interpolation for cold-climate COP and capacity curves.\n    """\n    # 1. Provide default NEEP/AHRI benchmark rating curves if none supplied\n    if cop_ratings is None:\n        # Standard cold-climate inverter profile (47°F, 17°F, 5°F, -22°F)\n        cop_ratings = {47.0: 3.60, 17.0: 2.40, 5.0: 1.80, -22.0: 1.00}\n\n    if capacity_ratings is None:\n        # Base 3-ton unit (36,000 BTU/hr nominal at 47°F) with cold weather capacity drop\n        capacity_ratings = {\n            47.0: hp_capacity_tons * 12000.0,\n            17.0: hp_capacity_tons * 12000.0 * 0.78,\n            5.0: hp_capacity_tons * 12000.0 * 0.60,\n            -22.0: hp_capacity_tons * 12000.0 * 0.30,\n        }\n\n    # Sort dictionary keys to ensure ascending arrays for interpolation\n    temps_cop = np.array(sorted(cop_ratings.keys()))\n    vals_cop = np.array([cop_ratings[t] for t in temps_cop])\n\n    temps_cap = np.array(sorted(capacity_ratings.keys()))\n    vals_cap = np.array([capacity_ratings[t] for t in temps_cap])\n\n    # 2. Build SciPy linear interpolation functions\n    get_cop_interp = interp1d(\n        temps_cop,\n        vals_cop,\n        kind=\'linear\',\n        bounds_error=False,\n        fill_value=(vals_cop[0], vals_cop[-1]),  # Clip at boundaries\n    )\n\n    get_cap_interp = interp1d(\n        temps_cap,\n        vals_cap,\n        kind=\'linear\',\n        bounds_error=False,\n        fill_value=(vals_cap[0], vals_cap[-1]),  # Clip at boundaries\n    )\n\n    # 3. Setup Building & Simulation State\n    n_hours = len(df_hourly)\n    ua_btu = area_sqft / r_val\n    c_btu = area_sqft * 15.0\n\n    t_in = np.zeros(n_hours)\n    q_hp_delivered = np.zeros(n_hours)\n    q_aux_delivered = np.zeros(n_hours)\n    kwh_hp = np.zeros(n_hours)\n    therms_aux = np.zeros(n_hours)\n    hvac_action = [\'OFF\'] * n_hours\n\n    t_in[0] = setpoint_f\n    dt_hr = 1.0\n\n    # 4. Hourly Simulation Loop\n    for t in range(n_hours - 1):\n        t_amb = df_hourly[\'t_amb_f\'].iloc[t]\n        q_int = df_hourly[\'q_int_btu\'].iloc[t]\n\n        heat_loss = ua_btu * (t_in[t] - t_amb)\n        heat_needed = max(0.0, heat_loss - q_int)\n\n        if t_in[t] < (setpoint_f - deadband_f) or heat_needed > 0:\n            # Look up interpolated COP and max capacity for current outdoor temp\n            cop_current = float(get_cop_interp(t_amb))\n            avail_hp_cap = float(get_cap_interp(t_amb))\n\n            # Heat delivered split\n            hp_heat = min(heat_needed, avail_hp_cap)\n            aux_heat = max(0.0, heat_needed - hp_heat)\n\n            # Energy inputs\n            hp_elec_kwh = (hp_heat / max(1.0, cop_current)) / 3412.14\n            gas_therms = (aux_heat / gas_furnace_eff) / 100000.0\n\n            if hp_heat > 0 and aux_heat > 0:\n                hvac_action[t] = \'HP_AND_GAS_AUX\'\n            elif hp_heat > 0:\n                hvac_action[t] = \'HP_ONLY\'\n            elif aux_heat > 0:\n                hvac_action[t] = \'GAS_AUX_ONLY\'\n        else:\n            hp_heat = 0.0\n            aux_heat = 0.0\n            hp_elec_kwh = 0.0\n            gas_therms = 0.0\n            hvac_action[t] = \'OFF\'\n\n        q_hp_delivered[t] = hp_heat\n        q_aux_delivered[t] = aux_heat\n        kwh_hp[t] = hp_elec_kwh\n        therms_aux[t] = gas_therms\n\n        # State update\n        net_heat_gain = hp_heat + aux_heat + q_int - heat_loss\n        t_in[t + 1] = t_in[t] + (net_heat_gain / c_btu) * dt_hr\n\n    hvac_action[-1] = hvac_action[-2]\n\n    # 5. Format Output Dataframe\n    df_res = df_hourly.copy()\n    df_res[\'t_in_f\'] = t_in\n    df_res[\'hvac_action\'] = hvac_action\n    df_res[\'q_hp_btu\'] = q_hp_delivered\n    df_res[\'q_aux_btu\'] = q_aux_delivered\n    df_res[\'kwh_hp\'] = kwh_hp\n    df_res[\'therms_aux\'] = therms_aux\n\n    df_res[\'elec_cost_usd\'] = df_res[\'kwh_hp\'] * elec_rate_per_kwh\n    df_res[\'gas_cost_usd\'] = df_res[\'therms_aux\'] * gas_rate_per_therm\n    df_res[\'total_cost_usd\'] = (\n        df_res[\'elec_cost_usd\'] + df_res[\'gas_cost_usd\']\n    )\n\n    df_res[\'co2_elec_kg\'] = df_res[\'kwh_hp\'] * grid_co2_kg_per_kwh\n    df_res[\'co2_gas_kg\'] = df_res[\'therms_aux\'] * gas_co2_kg_per_therm\n    df_res[\'total_co2_kg\'] = df_res[\'co2_elec_kg\'] + df_res[\'co2_gas_kg\']\n\n    summary = {\n        \'total_elec_kwh\': df_res[\'kwh_hp\'].sum(),\n        \'total_gas_therms\': df_res[\'therms_aux\'].sum(),\n        \'total_cost_usd\': df_res[\'total_cost_usd\'].sum(),\n        \'total_co2_kg\': df_res[\'total_co2_kg\'].sum(),\n        \'hp_coverage_pct\': (\n            df_res[\'q_hp_btu\'].sum()\n            / max(1.0, (df_res[\'q_hp_btu\'].sum() + df_res[\'q_aux_btu\'].sum()))\n        )\n        * 100,\n        \'min_indoor_temp_f\': df_res[\'t_in_f\'].min(),\n    }\n\n    return df_res, summary\n'

**Finding:** in the copied function the house has no cooling or venting. Internal gains push the indoor temperature up to ~90 F in mild and summer weather, and the loop keeps "heating" against that warm interior (`heat_needed` only checks loss vs. gains), so annual energy comes out well above the true heating need. Below, `run_thermal_hp_sim_capped` is the same code with one line changed: indoor temperature is capped at the setpoint (free cooling). The cross-check shows both.

In [ ]:
_old = "t_in[t + 1] = t_in[t] + (net_heat_gain / c_btu) * dt_hr"
_src = SIM_SOURCE
assert _old in _src
exec(_src.replace(_old, "t_in[t + 1] = min(setpoint_f, t_in[t] + (net_heat_gain / c_btu) * dt_hr)  # PATCH: cap at setpoint")
          .replace("def run_thermal_hp_sim(", "def run_thermal_hp_sim_capped("))

In [ ]:
hours_idx = pd.date_range("2026-01-01", periods=len(w), freq="h")
df_in = pd.DataFrame({"t_amb_f": w["t_f"].values, "q_int_btu": Q_INT_BTUH}, index=hours_idx)

def patrick_check(size_kbtuh, fn=None):
    fn = fn or run_thermal_hp_sim
    tons = size_kbtuh / 12
    caps = {t: size_kbtuh * 1000 * f for t, f in CAP_FRAC.items()}
    _, summ = fn(df_in, area_sqft=AREA_SQFT, r_val=R_VAL, setpoint_f=SETPOINT_F,
        cop_ratings=COP_PTS, capacity_ratings=caps, hp_capacity_tons=tons, gas_furnace_eff=FURNACE_AFUE,
        elec_rate_per_kwh=ELEC_USD_KWH, gas_rate_per_therm=GAS_USD_KWH_FUEL * 29.3071,
        grid_co2_kg_per_kwh=GRID_KG_KWH, gas_co2_kg_per_therm=GAS_KG_KWH_FUEL * 29.3071)
    return summ

rows = []
for _, r in res[res.dispatch == "hp_first"].drop_duplicates("size_kbtuh").iterrows():
    if r.size_kbtuh == 0: continue
    p, pc = patrick_check(r.size_kbtuh), patrick_check(r.size_kbtuh, run_thermal_hp_sim_capped)
    rows.append(dict(size_kbtuh=r.size_kbtuh, gurobi_elec_kwh=r.annual_elec_kwh, capped_elec_kwh=pc["total_elec_kwh"], orig_elec_kwh=p["total_elec_kwh"],
                     gurobi_gas_therms=r.annual_gas_kwh_fuel / 29.3071, capped_gas_therms=pc["total_gas_therms"], orig_gas_therms=p["total_gas_therms"],
                     gurobi_cost=r.annual_cost_usd, capped_cost=pc["total_cost_usd"], orig_cost=p["total_cost_usd"],
                     gurobi_co2_t=r.annual_co2_t, capped_co2_t=pc["total_co2_kg"]/1000, orig_co2_t=p["total_co2_kg"]/1000))
pd.DataFrame(rows).round(1)

## Reading the results / known gaps
- `optimal` dispatch lets Gurobi use gas in the coldest, lowest-COP hours even when the heat pump has spare capacity (like v1's smart controller); `hp_first` is Patrick's rule and is the one the cross-check compares against. `capped` = Patrick's function with indoor temperature capped at setpoint; `orig` = unchanged (see finding above).
- The binned model ignores indoor thermal mass and the setpoint deadband (Patrick's loop has both), so expect small differences in the cross-check table; large ones mean the binning is too coarse.
- Gas backup is a furnace at 95% AFUE (inputs file), not Patrick's 80% default.
- Time-of-use rates, the CenterPoint rule, AC credit, cold-weather scenario, defrost and cooling are not modeled here (v1 has some of them).
- Prices, rebate and the cost equation are placeholders from the repo's data files; verify before quoting.
- Emissions use an annual grid average, not hourly or marginal.